# 🐍 Module 3 — Python for ML Engineering
**ML Engineer Program**

| Section | Content |
|---|---|
| 3.1 | OOP applied to ML — BaseEstimator, TransformerMixin, ABC |
| 3.2 | Pipelines and composition — Pipeline, ColumnTransformer |
| 3.3 | Files, serialization, config — pathlib, json, joblib, dotenv |
| 3.4 | Logging and debugging — logging, pdb |

---
## 3.1 OOP applied to ML

### Lesson — BaseEstimator & TransformerMixin

In [1]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.utils.validation import check_is_fitted
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score

# ── Why inherit from BaseEstimator + TransformerMixin ─────────
# BaseEstimator   → get_params() / set_params() for free
#                   required for Pipeline + GridSearch/Optuna
# TransformerMixin → fit_transform() for free (calls fit then transform)

# ── Stateless transformer (no learned params) ─────────────────
class LogTransformer(BaseEstimator, TransformerMixin):
    """Apply log1p to all-positive columns, leave others unchanged."""

    def fit(self, X, y=None):
        X = np.asarray(X)
        # learn which columns are always non-negative
        self.positive_cols_ = np.all(X >= 0, axis=0)
        return self   # always return self

    def transform(self, X):
        check_is_fitted(self, "positive_cols_")
        X_out = np.asarray(X, dtype=float).copy()
        X_out[:, self.positive_cols_] = np.log1p(X_out[:, self.positive_cols_])
        return X_out

# ── Stateful transformer (learns from data) ───────────────────
class OutlierClipper(BaseEstimator, TransformerMixin):
    """Clip values to mean ± n_std * std, computed on train set."""

    def __init__(self, n_std: float = 3.0):
        self.n_std = n_std   # RULE: hyperparams must match __init__ param names exactly

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.mean_ = X.mean(axis=0)
        self.std_  = X.std(axis=0)
        return self

    def transform(self, X):
        check_is_fitted(self, ["mean_", "std_"])
        X = np.asarray(X, dtype=float)
        lower = self.mean_ - self.n_std * self.std_
        upper = self.mean_ + self.n_std * self.std_
        return np.clip(X, lower, upper)


class FeatureSelector(BaseEstimator, TransformerMixin):
    """Select a subset of features by index."""

    def __init__(self, indices: list):
        self.indices = indices

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = np.asarray(X)
        return X[:, self.indices]


# ── Verify sklearn compatibility ──────────────────────────────
X, y = load_breast_cancer(return_X_y=True)

pipe = Pipeline([
    ("clip",   OutlierClipper(n_std=3.0)),
    ("log",    LogTransformer()),
    ("select", FeatureSelector(indices=list(range(10)))),
    ("scale",  StandardScaler()),
    ("clf",    LogisticRegression(max_iter=1000)),
])

scores = cross_val_score(pipe, X, y, cv=5, scoring="roc_auc")
print(f"ROC-AUC: {scores.mean():.4f} ± {scores.std():.4f}")

# get_params / set_params — required for hyperparameter tuning
print("\nget_params:", pipe["clip"].get_params())

# clone — deep copy without fitted state
clipper = OutlierClipper(n_std=2.5)
clipper.fit(X)
cloned = clone(clipper)
print("cloned n_std:", cloned.n_std)
try:
    cloned.transform(X)   # not fitted — should raise
except Exception as e:
    print(f"not fitted: {type(e).__name__}")

# KEY INSIGHT:
# __init__ params must have the SAME name as the stored attribute
# → BaseEstimator.get_params() introspects __init__ signature
# → if you store self.n_std_ (with underscore) but __init__ has n_std → bug
# Learned attributes (from fit) end with underscore: mean_, std_, etc.

ROC-AUC: 0.9871 ± 0.0072

get_params: {'n_std': 3.0}
cloned n_std: 2.5
not fitted: NotFittedError


### Lesson — Abstract Base Classes

In [2]:
from abc import ABC, abstractmethod
from typing import Optional
import numpy as np

# ── Why ABC ───────────────────────────────────────────────────
# Enforces a contract: any subclass MUST implement abstract methods
# Prevents instantiation of incomplete classes
# Documents the expected interface clearly

class BasePredictor(ABC):
    """Contract: any ML predictor must implement these methods."""

    @abstractmethod
    def fit(self, X: np.ndarray, y: np.ndarray) -> "BasePredictor":
        """Fit model on training data. Must return self."""
        ...

    @abstractmethod
    def predict(self, X: np.ndarray) -> np.ndarray:
        """Return predictions for X."""
        ...

    @abstractmethod
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Return class probabilities. Shape: (n_samples, n_classes)."""
        ...

    # Concrete method — shared by all subclasses
    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Accuracy score — no need to override in most cases."""
        return (self.predict(X) == y).mean()


class ThresholdClassifier(BasePredictor):
    """Classifies based on a single feature threshold."""

    def __init__(self, feature_idx: int = 0, threshold: Optional[float] = None):
        self.feature_idx = feature_idx
        self.threshold   = threshold

    def fit(self, X: np.ndarray, y: np.ndarray) -> "ThresholdClassifier":
        col = X[:, self.feature_idx]
        # find threshold that minimizes misclassification
        candidates = np.unique(col)
        best_acc, best_t = 0.0, candidates[0]
        for t in candidates:
            preds = (col > t).astype(int)
            acc   = (preds == y).mean()
            if acc > best_acc:
                best_acc, best_t = acc, t
        self.threshold_ = best_t
        self.train_acc_  = best_acc
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        return (X[:, self.feature_idx] > self.threshold_).astype(int)

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        col = X[:, self.feature_idx]
        # simple sigmoid-like scaling
        p = 1 / (1 + np.exp(-(col - self.threshold_)))
        return np.column_stack([1 - p, p])


# Can't instantiate ABC directly
try:
    BasePredictor()
except TypeError as e:
    print(f"Cannot instantiate ABC: {e}")

# Works with concrete implementation
from sklearn.datasets import load_breast_cancer
X, y = load_breast_cancer(return_X_y=True)

clf = ThresholdClassifier(feature_idx=0)
clf.fit(X, y)
print(f"\nThresholdClassifier accuracy: {clf.score(X, y):.4f}")
print(f"Threshold found: {clf.threshold_:.4f}")
print(f"Probabilities sample: {clf.predict_proba(X[:3]).round(3)}")

Cannot instantiate ABC: Can't instantiate abstract class BasePredictor without an implementation for abstract methods 'fit', 'predict', 'predict_proba'

ThresholdClassifier accuracy: 0.6257
Threshold found: 6.9810
Probabilities sample: [[0. 1.]
 [0. 1.]
 [0. 1.]]


### 🏋️ Exercises 3.1

In [3]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 — Custom transformer: RatioFeatures
# ════════════════════════════════════════════════════════
# Implement RatioFeatures(pairs) where pairs is a list of
# (i, j) tuples. For each pair, the transformer adds a new
# column = X[:, i] / (X[:, j] + epsilon) to avoid division by 0.
# The output has shape (n_samples, n_features + len(pairs)).
# Must be sklearn-compatible (BaseEstimator + TransformerMixin).

import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score

class RatioFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, pairs: list, epsilon: float = 1e-8):
        self.pairs   = pairs
        self.epsilon = epsilon

    def fit(self, X, y=None):
        pass  # complete here

    def transform(self, X):
        pass  # complete here

# Test
X, y = load_breast_cancer(return_X_y=True)
pairs = [(0, 1), (2, 3), (4, 5)]

pipe = Pipeline([
    ("ratios", RatioFeatures(pairs=pairs)),
    ("scale",  StandardScaler()),
    ("clf",    LogisticRegression(max_iter=1000)),
])

scores = cross_val_score(pipe, X, y, cv=5, scoring="roc_auc")
print(f"With RatioFeatures — ROC-AUC: {scores.mean():.4f} ± {scores.std():.4f}")

# get_params must work
rf = RatioFeatures(pairs=[(0,1)])
print("get_params:", rf.get_params())

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def fit(self, X, y=None):
#     self.n_features_in_ = np.asarray(X).shape[1]
#     return self
#
# def transform(self, X):
#     check_is_fitted(self, "n_features_in_")
#     X = np.asarray(X, dtype=float)
#     ratios = np.column_stack([
#         X[:, i] / (X[:, j] + self.epsilon)
#         for i, j in self.pairs
#     ])
#     return np.hstack([X, ratios])

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\adrie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\adrie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "C:\Users\adrie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0

In [4]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 — ABC: BaseMonitor with concrete subclasses
# ════════════════════════════════════════════════════════
# Implement the abstract class BaseMonitor with:
# - abstract method check(data) → dict of metrics
# - abstract method is_alert(metrics) → bool
# - concrete method run(data) → calls check + is_alert,
#   prints a summary, returns (metrics, alert_triggered)
#
# Then implement two concrete monitors:
# a) MissingValueMonitor(threshold=0.05)
#    → check: {"missing_rate": float per column}
#    → alert: if any column > threshold
# b) DriftMonitor(reference_mean, threshold=0.1)
#    → check: {"mean_drift": abs(current_mean - ref_mean) / ref_mean per col}
#    → alert: if any column drift > threshold

from abc import ABC, abstractmethod
import numpy as np
from typing import Dict, Tuple

class BaseMonitor(ABC):
    pass  # complete here

class MissingValueMonitor(BaseMonitor):
    pass  # complete here

class DriftMonitor(BaseMonitor):
    pass  # complete here

# Tests
import numpy as np
np.random.seed(42)

X_ref  = np.random.randn(200, 4)
X_prod = np.random.randn(100, 4)
X_prod[:10, 2] = np.nan          # inject missing
X_prod[:, 3]  += 2.0             # inject drift on col 3

mv_monitor = MissingValueMonitor(threshold=0.05)
metrics, alert = mv_monitor.run(X_prod)
print(f"MissingValue alert: {alert}")

drift_monitor = DriftMonitor(reference_mean=X_ref.mean(axis=0), threshold=0.1)
metrics, alert = drift_monitor.run(X_prod)
print(f"Drift alert: {alert}")

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# class BaseMonitor(ABC):
#     @abstractmethod
#     def check(self, data: np.ndarray) -> Dict[str, np.ndarray]: ...
#     @abstractmethod
#     def is_alert(self, metrics: Dict) -> bool: ...
#     def run(self, data: np.ndarray) -> Tuple[Dict, bool]:
#         metrics = self.check(data)
#         alert   = self.is_alert(metrics)
#         status  = "🔴 ALERT" if alert else "🟢 OK"
#         print(f"[{self.__class__.__name__}] {status} — {metrics}")
#         return metrics, alert
#
# class MissingValueMonitor(BaseMonitor):
#     def __init__(self, threshold=0.05): self.threshold = threshold
#     def check(self, data):
#         return {"missing_rate": np.isnan(data).mean(axis=0)}
#     def is_alert(self, metrics):
#         return bool(np.any(metrics["missing_rate"] > self.threshold))
#
# class DriftMonitor(BaseMonitor):
#     def __init__(self, reference_mean, threshold=0.1):
#         self.reference_mean = np.asarray(reference_mean)
#         self.threshold = threshold
#     def check(self, data):
#         cur = np.nanmean(data, axis=0)
#         drift = np.abs(cur - self.reference_mean) / (np.abs(self.reference_mean) + 1e-8)
#         return {"mean_drift": drift}
#     def is_alert(self, metrics):
#         return bool(np.any(metrics["mean_drift"] > self.threshold))

TypeError: MissingValueMonitor() takes no arguments

---
## 3.2 Pipelines and composition

### Lesson — Pipeline, ColumnTransformer, FunctionTransformer

In [5]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.preprocessing import (StandardScaler, OneHotEncoder,
                                    OrdinalEncoder, FunctionTransformer)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.datasets import fetch_openml
import lightgbm as lgb

np.random.seed(42)
n = 1000

df = pd.DataFrame({
    "age":         np.random.randint(18, 70, n).astype(float),
    "income":      np.random.exponential(50000, n),
    "tenure":      np.random.randint(0, 60, n).astype(float),
    "charges":     np.random.normal(65, 30, n).clip(10, 200),
    "nb_products": np.random.randint(1, 6, n).astype(float),
    "contract":    np.random.choice(["month-to-month","one-year","two-year"], n),
    "region":      np.random.choice(["IDF","PACA","AURA","HDF"], n),
    "missing_inc": np.where(np.random.rand(n) < 0.15, np.nan,
                            np.random.exponential(30000, n)),
})
# inject missing values
df.loc[np.random.choice(n, 50, replace=False), "age"]    = np.nan
df.loc[np.random.choice(n, 30, replace=False), "tenure"] = np.nan

churn_p = (0.3 - 0.004*df["tenure"].fillna(30)
           + 0.002*np.random.randn(n)).clip(0, 1)
y = (churn_p > 0.28).astype(int)

X = df.copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y,
                                                      test_size=0.2, random_state=42)

# ── ColumnTransformer — handle mixed types ────────────────────
num_features = ["age", "income", "tenure", "charges", "nb_products", "missing_inc"]
cat_ohe      = ["region"]
cat_ord      = ["contract"]

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

cat_ohe_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe",     OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

cat_ord_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ord",     OrdinalEncoder(categories=[["month-to-month","one-year","two-year"]])),
])

preprocessor = ColumnTransformer([
    ("num",     num_pipe,     num_features),
    ("cat_ohe", cat_ohe_pipe, cat_ohe),
    ("cat_ord", cat_ord_pipe, cat_ord),
], remainder="drop")

# ── Full pipeline ─────────────────────────────────────────────
full_pipe = Pipeline([
    ("prep", preprocessor),
    ("clf",  lgb.LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)),
])

scores = cross_val_score(full_pipe, X_train, y_train, cv=5, scoring="roc_auc")
print(f"ROC-AUC CV: {scores.mean():.4f} ± {scores.std():.4f}")

full_pipe.fit(X_train, y_train)
print(f"Test ROC-AUC: {full_pipe.score(X_test, y_test):.4f}")

# ── FunctionTransformer — wrap a lambda or function ──────────
log_transformer = FunctionTransformer(np.log1p, validate=True)
X_demo = np.array([[1.0, 100.0], [2.0, 200.0]])
print("\nFunctionTransformer log1p:", log_transformer.transform(X_demo))

# ── Accessing steps ───────────────────────────────────────────
print("\nPipeline steps:", [name for name, _ in full_pipe.steps])
print("Preprocessor transformers:",
      [name for name, _, _ in full_pipe["prep"].transformers])

# ── GridSearchCV on pipeline params ──────────────────────────
param_grid = {
    "clf__n_estimators":  [50, 100],
    "clf__num_leaves":    [31, 63],
    "prep__num__imputer__strategy": ["mean", "median"],
}
gs = GridSearchCV(full_pipe, param_grid, cv=3, scoring="roc_auc", n_jobs=-1)
gs.fit(X_train, y_train)
print(f"\nBest params: {gs.best_params_}")
print(f"Best CV score: {gs.best_score_:.4f}")

C:\Users\adrie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\adrie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\adrie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\adrie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-pac

ROC-AUC CV: 0.9985 ± 0.0019
Test ROC-AUC: 1.0000

FunctionTransformer log1p: [[0.69314718 4.61512052]
 [1.09861229 5.30330491]]

Pipeline steps: ['prep', 'clf']
Preprocessor transformers: ['num', 'cat_ohe', 'cat_ord']


C:\Users\adrie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Best params: {'clf__n_estimators': 50, 'clf__num_leaves': 31, 'prep__num__imputer__strategy': 'mean'}
Best CV score: 0.9986


### 🏋️ Exercises 3.2

In [3]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 — Build a full pipeline from scratch
# ════════════════════════════════════════════════════════
# Given the dataset below, build a complete sklearn Pipeline that:
# 1. Handles missing values (median for numeric, most_frequent for cat)
# 2. Applies log1p to income and charges (skewed distributions)
#    HINT: use FunctionTransformer inside the numeric sub-pipeline
# 3. Scales numeric features
# 4. One-hot encodes categorical features
# 5. Uses LightGBM as classifier with class_weight="balanced"
#
# Evaluate with 5-fold stratified CV on ROC-AUC.
# Then show the feature names out of the preprocessor.

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score, StratifiedKFold
import lightgbm as lgb

np.random.seed(42)
n = 800
df = pd.DataFrame({
    "age":      np.where(np.random.rand(n)<0.08, np.nan, np.random.randint(18,70,n).astype(float)),
    "income":   np.where(np.random.rand(n)<0.12, np.nan, np.random.exponential(50000,n)),
    "tenure":   np.random.randint(0, 60, n).astype(float),
    "charges":  np.random.exponential(80, n).clip(10, 300),
    "contract": np.random.choice(["monthly","annual","biennial"], n),
    "region":   np.where(np.random.rand(n)<0.05, None,
                        np.random.choice(["IDF","PACA","AURA"], n)),
})
churn = (np.random.rand(n) < 0.25).astype(int)

# complete here
num_cols = ["age", "income", "tenure", "charges"]
cat_cols = ["contract", "region"]

pipe = None  # build your pipeline here

if pipe is not None:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(pipe, df, churn, cv=cv, scoring="roc_auc")
    print(f"ROC-AUC: {scores.mean():.4f} ± {scores.std():.4f}")

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# log_cols = ["income", "charges"]
# other_num = ["age", "tenure"]
#
# num_pipe = Pipeline([
#     ("imputer", SimpleImputer(strategy="median")),
#     ("log",     FunctionTransformer(np.log1p, validate=True)),
#     ("scaler",  StandardScaler()),
# ])
# cat_pipe = Pipeline([
#     ("imputer", SimpleImputer(strategy="most_frequent")),
#     ("ohe",     OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
# ])
# prep = ColumnTransformer([
#     ("num", num_pipe, num_cols),
#     ("cat", cat_pipe, cat_cols),
# ])
# pipe = Pipeline([
#     ("prep", prep),
#     ("clf",  lgb.LGBMClassifier(100, class_weight="balanced",
#                                  random_state=42, verbose=-1)),
# ])

In [8]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 — set_params and clone
# ════════════════════════════════════════════════════════
# Given this pipeline, demonstrate:
# a) Use set_params to change clf__n_estimators to 200
#    and prep__num__imputer__strategy to "mean"
# b) clone the pipeline (unfitted copy)
# c) Show that the clone has the updated params but is not fitted
# d) Run a quick cross_val_score on both original and modified

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.datasets import load_breast_cancer
import lightgbm as lgb

X, y = load_breast_cancer(return_X_y=True)

prep = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
    ]), list(range(30))),
])
pipe = Pipeline([
    ("prep", prep),
    ("clf",  lgb.LGBMClassifier(n_estimators=50, random_state=42, verbose=-1)),
])

# a) set_params — complete here
# b) clone — complete here
# c) verify — complete here
# d) compare — complete here

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# # a)
# pipe.set_params(clf__n_estimators=200,
#                 prep__num__imputer__strategy="mean")
# print("after set_params:", pipe.get_params()["clf__n_estimators"])
#
# # b)
# pipe_clone = clone(pipe)
# print("clone n_estimators:", pipe_clone.get_params()["clf__n_estimators"])
#
# # c)
# from sklearn.exceptions import NotFittedError
# try:
#     pipe_clone["clf"].predict(X)
# except NotFittedError as e:
#     print("clone not fitted:", e)
#
# # d)
# from sklearn.model_selection import cross_val_score
# s_orig  = cross_val_score(pipe,       X, y, cv=3, scoring="roc_auc").mean()
# s_clone = cross_val_score(pipe_clone, X, y, cv=3, scoring="roc_auc").mean()
# print(f"original: {s_orig:.4f} | clone: {s_clone:.4f}")

---
## 3.3 Files, serialization, config

### Lesson — pathlib, json, yaml, joblib, dotenv

In [9]:
import json
import os
import tempfile
from pathlib import Path
import joblib
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris

# ── pathlib — the right way to handle paths ──────────────────
# Never use os.path.join or string concatenation for paths

base_dir  = Path("/tmp/ml_project")
data_dir  = base_dir / "data" / "raw"
model_dir = base_dir / "models"
log_dir   = base_dir / "logs"

# Create directory tree
for d in [data_dir, model_dir, log_dir]:
    d.mkdir(parents=True, exist_ok=True)

print("Project structure:")
for p in sorted(base_dir.rglob("*")):
    indent = "  " * (len(p.parts) - len(base_dir.parts) - 1)
    print(f"  {indent}{p.name}/")

# Common pathlib operations
config_path = base_dir / "config.json"
print(f"\nstem      : {config_path.stem}")     # config
print(f"suffix    : {config_path.suffix}")     # .json
print(f"parent    : {config_path.parent}")     # /tmp/ml_project
print(f"exists    : {config_path.exists()}")   # False

# Write / read text
config_path.write_text('{"model": "lgbm", "version": "1.0"}')
content = config_path.read_text()
print(f"read_text : {content}")

# Glob
(base_dir / "data" / "raw" / "train.csv").write_text("a,b\n1,2")
(base_dir / "data" / "raw" / "test.csv").write_text("a,b\n3,4")
csv_files = list(base_dir.rglob("*.csv"))
print(f"\nCSV files : {[f.name for f in csv_files]}")

# ── json — configs and metadata ──────────────────────────────
experiment = {
    "model":       "lightgbm",
    "version":     "v1.2.0",
    "metrics":     {"roc_auc": 0.9412, "f1": 0.8731},
    "params":      {"n_estimators": 200, "learning_rate": 0.05},
    "features":    ["age", "tenure", "charges"],
    "trained_at":  "2026-03-18T10:00:00",
}

json_path = model_dir / "experiment.json"
json_path.write_text(json.dumps(experiment, indent=2))
loaded = json.loads(json_path.read_text())
print(f"\nJSON round-trip OK: {loaded['metrics']['roc_auc']}")

# ── joblib — serialize sklearn models ────────────────────────
X, y = load_iris(return_X_y=True)
model = LogisticRegression(max_iter=1000).fit(X, y)

model_path = model_dir / "model_v1.pkl"
joblib.dump(model, model_path, compress=3)   # compress=3 → smaller file
loaded_model = joblib.load(model_path)

preds_orig   = model.predict(X[:5])
preds_loaded = loaded_model.predict(X[:5])
print(f"\nModel round-trip OK: {(preds_orig == preds_loaded).all()}")
print(f"Model file size: {model_path.stat().st_size} bytes")

# Save with metadata
bundle = {"model": model, "metadata": experiment, "feature_names": ["a","b","c","d"]}
joblib.dump(bundle, model_dir / "model_bundle.pkl")
loaded_bundle = joblib.load(model_dir / "model_bundle.pkl")
print(f"Bundle version: {loaded_bundle['metadata']['version']}")

# ── environment variables with os.environ ─────────────────────
os.environ["MODEL_VERSION"] = "v1.2.0"
os.environ["BATCH_SIZE"]    = "512"

version    = os.environ.get("MODEL_VERSION", "unknown")
batch_size = int(os.environ.get("BATCH_SIZE", "256"))
missing    = os.environ.get("MISSING_KEY", "default_value")

print(f"\nENV MODEL_VERSION: {version}")
print(f"ENV BATCH_SIZE    : {batch_size}")
print(f"ENV MISSING_KEY   : {missing}")

Project structure:
  data/
    raw/
  logs/
  models/

stem      : config
suffix    : .json
parent    : \tmp\ml_project
exists    : False
read_text : {"model": "lgbm", "version": "1.0"}

CSV files : ['test.csv', 'train.csv']

JSON round-trip OK: 0.9412

Model round-trip OK: True
Model file size: 696 bytes
Bundle version: v1.2.0

ENV MODEL_VERSION: v1.2.0
ENV BATCH_SIZE    : 512
ENV MISSING_KEY   : default_value


### Lesson — Config patterns for ML projects

In [10]:
import json
import os
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import List, Dict, Optional
from pydantic import BaseModel, Field

# ── Pattern 1: Pydantic config loaded from JSON/env ──────────
class MLConfig(BaseModel):
    """Full ML pipeline configuration — validated at load time."""
    model_name:    str
    n_estimators:  int   = Field(default=100, ge=1)
    learning_rate: float = Field(default=0.05, gt=0)
    num_leaves:    int   = Field(default=31, ge=2)
    features:      List[str]
    target_col:    str   = "churn"
    test_size:     float = Field(default=0.2, gt=0, lt=1)
    random_state:  int   = 42

    @classmethod
    def from_json(cls, path: Path) -> "MLConfig":
        return cls(**json.loads(path.read_text()))

    @classmethod
    def from_env(cls, prefix: str = "ML_") -> "MLConfig":
        """Load config from environment variables with given prefix."""
        env_vars = {
            k[len(prefix):].lower(): v
            for k, v in os.environ.items()
            if k.startswith(prefix)
        }
        return cls(**env_vars)

    def to_json(self, path: Path) -> None:
        path.write_text(self.model_dump_json(indent=2))


# Save and reload
cfg = MLConfig(
    model_name="lgbm_churn_v1",
    n_estimators=200,
    learning_rate=0.05,
    features=["age", "tenure", "charges", "nb_products"],
)

config_path = Path("/tmp/config.json")
cfg.to_json(config_path)
cfg_loaded = MLConfig.from_json(config_path)
print("Config round-trip OK:", cfg == cfg_loaded)
print(cfg_loaded.model_dump_json(indent=2))

# ── Pattern 2: versioned model registry ──────────────────────
class ModelRegistry:
    """Simple file-based model registry."""

    def __init__(self, registry_dir: Path):
        self.registry_dir = registry_dir
        self.registry_dir.mkdir(parents=True, exist_ok=True)
        self._index_path = registry_dir / "index.json"
        self._index: Dict = self._load_index()

    def _load_index(self) -> dict:
        if self._index_path.exists():
            return json.loads(self._index_path.read_text())
        return {"models": {}}

    def _save_index(self) -> None:
        self._index_path.write_text(json.dumps(self._index, indent=2))

    def register(self, name: str, version: str,
                 model, metadata: dict) -> Path:
        import joblib
        model_path = self.registry_dir / f"{name}_{version}.pkl"
        joblib.dump({"model": model, "metadata": metadata}, model_path)
        self._index["models"][f"{name}@{version}"] = {
            "path":     str(model_path),
            "metadata": metadata,
        }
        self._save_index()
        print(f"Registered {name}@{version}")
        return model_path

    def load(self, name: str, version: str) -> dict:
        import joblib
        key = f"{name}@{version}"
        if key not in self._index["models"]:
            raise KeyError(f"Model {key} not found in registry")
        return joblib.load(self._index["models"][key]["path"])

    def list_models(self) -> List[str]:
        return list(self._index["models"].keys())


from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris

registry = ModelRegistry(Path("/tmp/model_registry"))
X, y = load_iris(return_X_y=True)
model = LogisticRegression(max_iter=1000).fit(X, y)

registry.register("iris_clf", "v1.0", model,
                   metadata={"accuracy": 0.97, "features": 4})
print("Registry:", registry.list_models())

bundle = registry.load("iris_clf", "v1.0")
print("Loaded accuracy:", bundle["metadata"]["accuracy"])

Config round-trip OK: True
{
  "model_name": "lgbm_churn_v1",
  "n_estimators": 200,
  "learning_rate": 0.05,
  "num_leaves": 31,
  "features": [
    "age",
    "tenure",
    "charges",
    "nb_products"
  ],
  "target_col": "churn",
  "test_size": 0.2,
  "random_state": 42
}
Registered iris_clf@v1.0
Registry: ['iris_clf@v1.0']
Loaded accuracy: 0.97


### 🏋️ Exercises 3.3

In [11]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 — pathlib: project scaffolding function
# ════════════════════════════════════════════════════════
# Write create_project(base_path, project_name) that:
# - Creates the directory structure:
#     {project_name}/
#       data/raw/
#       data/processed/
#       models/
#       logs/
#       configs/
# - Writes a default config.json in configs/ with:
#     {"project": project_name, "version": "0.1.0",
#      "features": [], "target": "label"}
# - Returns a dict of all created paths
# - Uses pathlib throughout (no os.path)

from pathlib import Path
from typing import Dict
import json

def create_project(base_path: Path, project_name: str) -> Dict[str, Path]:
    pass  # complete here

paths = create_project(Path("/tmp"), "churn_project")
if paths:
    for name, path in paths.items():
        print(f"{name:15s}: {path}")
    config = json.loads(paths["config"].read_text())
    print("Default config:", config)

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def create_project(base_path, project_name):
#     root       = base_path / project_name
#     dirs = {
#         "data_raw":   root / "data" / "raw",
#         "data_proc":  root / "data" / "processed",
#         "models":     root / "models",
#         "logs":       root / "logs",
#         "configs":    root / "configs",
#     }
#     for d in dirs.values():
#         d.mkdir(parents=True, exist_ok=True)
#     config_path = dirs["configs"] / "config.json"
#     config_path.write_text(json.dumps({
#         "project": project_name,
#         "version": "0.1.0",
#         "features": [],
#         "target": "label",
#     }, indent=2))
#     return {**dirs, "config": config_path, "root": root}

In [12]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 — Experiment tracker
# ════════════════════════════════════════════════════════
# Write ExperimentTracker class that:
# - __init__(log_dir: Path): creates the directory, keeps a list of runs
# - log_run(name, params, metrics, model=None): saves a JSON file
#   named {name}_{timestamp}.json with params + metrics,
#   optionally serializes model with joblib to a .pkl file
# - get_best(metric: str) -> dict: returns the run with highest metric value
# - summary() -> pd.DataFrame: returns all runs as a DataFrame,
#   sorted by a key metric descending

from pathlib import Path
from typing import Dict, Any, Optional
import json, joblib, pandas as pd
from datetime import datetime

class ExperimentTracker:
    def __init__(self, log_dir: Path):
        pass  # complete here

    def log_run(self, name: str, params: dict, metrics: dict,
                model=None) -> Path:
        pass  # complete here

    def get_best(self, metric: str) -> dict:
        pass  # complete here

    def summary(self) -> pd.DataFrame:
        pass  # complete here

# Tests
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score

tracker = ExperimentTracker(Path("/tmp/experiments"))
X, y   = load_breast_cancer(return_X_y=True)

for name, model, params in [
    ("logreg", LogisticRegression(C=1.0, max_iter=1000),
     {"C": 1.0, "max_iter": 1000}),
    ("rf_100", RandomForestClassifier(n_estimators=100, random_state=42),
     {"n_estimators": 100}),
    ("rf_50",  RandomForestClassifier(n_estimators=50, random_state=42),
     {"n_estimators": 50}),
]:
    auc = cross_val_score(model, X, y, cv=3, scoring="roc_auc").mean()
    model.fit(X, y)
    if tracker is not None:
        tracker.log_run(name, params, {"roc_auc": round(auc, 4)}, model=model)

if tracker is not None:
    best = tracker.get_best("roc_auc")
    print("Best run:", best.get("name"), best.get("metrics"))
    print("\nSummary:")
    print(tracker.summary())

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# class ExperimentTracker:
#     def __init__(self, log_dir):
#         self.log_dir = Path(log_dir)
#         self.log_dir.mkdir(parents=True, exist_ok=True)
#         self._runs = []
#
#     def log_run(self, name, params, metrics, model=None):
#         ts  = datetime.now().strftime("%Y%m%d_%H%M%S")
#         run = {"name": name, "timestamp": ts,
#                "params": params, "metrics": metrics}
#         json_path = self.log_dir / f"{name}_{ts}.json"
#         json_path.write_text(json.dumps(run, indent=2))
#         if model is not None:
#             pkl_path = self.log_dir / f"{name}_{ts}.pkl"
#             joblib.dump(model, pkl_path)
#             run["model_path"] = str(pkl_path)
#         self._runs.append(run)
#         return json_path
#
#     def get_best(self, metric):
#         return max(self._runs, key=lambda r: r["metrics"].get(metric, -1))
#
#     def summary(self):
#         rows = []
#         for r in self._runs:
#             row = {"name": r["name"], "timestamp": r["timestamp"]}
#             row.update(r["params"])
#             row.update(r["metrics"])
#             rows.append(row)
#         df = pd.DataFrame(rows)
#         metric_cols = [c for c in df.columns if c not in ["name","timestamp"] + list(self._runs[0]["params"].keys())]
#         if metric_cols:
#             df = df.sort_values(metric_cols[0], ascending=False)
#         return df

C:\Users\adrie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\adrie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase t

AttributeError: 'NoneType' object has no attribute 'get'

---
## 3.4 Logging and debugging

### Course — logging module

In [13]:
import logging
import sys
from pathlib import Path

# ── Why logging, not print ────────────────────────────────────
# print → no level, no timestamp, no routing, can't disable in prod
# logging → levels, timestamps, multiple handlers, configurable per module

# ── Basic configuration ───────────────────────────────────────
# LEVELS: DEBUG(10) < INFO(20) < WARNING(30) < ERROR(40) < CRITICAL(50)

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(name)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
)

logger = logging.getLogger("ml_pipeline")
logger.debug("debug: detailed internals — disabled in prod")
logger.info("info: normal flow — model loaded, training started")
logger.warning("warning: unexpected but non-fatal — missing values detected")
logger.error("error: something failed — prediction returned NaN")
logger.critical("critical: system cannot continue")

# ── Module-level logger (best practice) ──────────────────────
# In each module/file: logger = logging.getLogger(__name__)
# This creates a logger named after the module → easy to filter

class MLPipeline:
    def __init__(self, name: str):
        self.name   = name
        self.logger = logging.getLogger(f"ml_pipeline.{name}")

    def fit(self, X, y):
        self.logger.info(f"Starting fit: {len(X)} samples, {X.shape[1]} features")
        try:
            import numpy as np
            self.mean_ = X.mean(axis=0)
            self.logger.debug(f"Computed mean: shape={self.mean_.shape}")
            self.logger.info("Fit complete")
        except Exception as e:
            self.logger.error(f"Fit failed: {e}", exc_info=True)
            raise
        return self

import numpy as np
X = np.random.randn(100, 5)
y = np.random.randint(0, 2, 100)
pipe = MLPipeline("churn_v1")
pipe.fit(X, y)

# ── File + console handler ────────────────────────────────────
def setup_logger(name: str, log_file: Path,
                 level: int = logging.INFO) -> logging.Logger:
    """Create a logger that writes to both console and file."""
    logger = logging.getLogger(name)
    logger.setLevel(level)
    logger.handlers.clear()   # avoid duplicate handlers on re-run

    fmt = logging.Formatter(
        "%(asctime)s | %(name)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    # Console handler — INFO and above
    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)

    # File handler — DEBUG and above
    fh = logging.FileHandler(log_file)
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)

    logger.addHandler(ch)
    logger.addHandler(fh)
    return logger

log_path = Path("/tmp/training.log")
train_logger = setup_logger("training", log_path)
train_logger.info("Training started")
train_logger.debug("This goes to file only")
train_logger.warning("High missing rate detected: 18%")

print(f"\nLog file content:")
print(log_path.read_text())

# ── Structured logging for ML ─────────────────────────────────
def log_metrics(logger: logging.Logger, step: str, **metrics) -> None:
    """Log metrics in a structured, parseable format."""
    metrics_str = " | ".join(f"{k}={v:.4f}" if isinstance(v, float)
                              else f"{k}={v}" for k, v in metrics.items())
    logger.info(f"[{step}] {metrics_str}")

log_metrics(train_logger, "epoch_1", loss=0.4231, auc=0.8912, lr=0.05)
log_metrics(train_logger, "epoch_2", loss=0.3871, auc=0.9134, lr=0.05)

15:43:49 | ml_pipeline | DEBUG | debug: detailed internals — disabled in prod
15:43:49 | ml_pipeline | INFO | info: normal flow — model loaded, training started
15:43:49 | ml_pipeline | WARNING | warning: unexpected but non-fatal — missing values detected
15:43:49 | ml_pipeline | ERROR | error: something failed — prediction returned NaN
15:43:49 | ml_pipeline | CRITICAL | critical: system cannot continue
15:43:49 | ml_pipeline.churn_v1 | INFO | Starting fit: 100 samples, 5 features
15:43:49 | ml_pipeline.churn_v1 | DEBUG | Computed mean: shape=(5,)
15:43:49 | ml_pipeline.churn_v1 | INFO | Fit complete
2026-03-19 15:43:49 | training | INFO | Training started
15:43:49 | training | INFO | Training started
2026-03-19 15:43:49 | training | WARNING | High missing rate detected: 18%
15:43:49 | training | WARNING | High missing rate detected: 18%

Log file content:
2026-03-19 15:43:49 | training | INFO | Training started
2026-03-19 15:43:49 | training | WARNING | High missing rate detected: 18

### 🏋️ Exercises 3.4

In [14]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 — Decorator: log_call
# ════════════════════════════════════════════════════════
# Write a decorator log_call(logger) that:
# - Logs "CALL {func_name}({args}, {kwargs})" at DEBUG level before call
# - Logs "RETURN {func_name} -> {result} ({elapsed:.1f}ms)" at INFO level after
# - Logs "ERROR {func_name}: {exception}" at ERROR level if exception raised,
#   then re-raises

import logging, time, functools, sys

logging.basicConfig(level=logging.DEBUG, stream=sys.stdout,
                    format="%(levelname)s | %(message)s")
logger = logging.getLogger("exercise")

def log_call(logger: logging.Logger):
    pass  # complete here

@log_call(logger)
def train_model(n_estimators: int, learning_rate: float) -> float:
    time.sleep(0.05)
    return round(0.9 + learning_rate * 0.1, 4)

@log_call(logger)
def failing_function(x: int) -> int:
    raise ValueError(f"Invalid input: {x}")

result = train_model(100, 0.05)
try:
    failing_function(-1)
except ValueError:
    pass

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def log_call(logger):
#     def decorator(func):
#         @functools.wraps(func)
#         def wrapper(*args, **kwargs):
#             logger.debug(f"CALL {func.__name__}(args={args}, kwargs={kwargs})")
#             t0 = time.perf_counter()
#             try:
#                 result = func(*args, **kwargs)
#                 elapsed = (time.perf_counter() - t0) * 1000
#                 logger.info(f"RETURN {func.__name__} -> {result} ({elapsed:.1f}ms)")
#                 return result
#             except Exception as e:
#                 elapsed = (time.perf_counter() - t0) * 1000
#                 logger.error(f"ERROR {func.__name__}: {e} ({elapsed:.1f}ms)")
#                 raise
#         return wrapper
#     return decorator

TypeError: 'NoneType' object is not callable

In [15]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 — Full pipeline with logging + error handling
# ════════════════════════════════════════════════════════
# Write run_training_pipeline(X, y, config) that:
# - Uses the setup_logger function from the course
# - Logs each step: validation, preprocessing, training, evaluation
# - Raises DataValidationError if X has NaN
# - Raises ModelTrainingError if cross-val score < config["min_auc"]
# - Returns a dict with model, metrics, and log_path
# - All exceptions must be logged before re-raising

import numpy as np
import logging
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

class DataValidationError(Exception): pass
class ModelTrainingError(Exception):  pass

def setup_logger(name, log_file, level=logging.INFO):
    logger = logging.getLogger(name)
    logger.setLevel(level)
    logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s",
                             datefmt="%H:%M:%S")
    ch = logging.StreamHandler()
    ch.setFormatter(fmt)
    logger.addHandler(ch)
    return logger

def run_training_pipeline(X: np.ndarray, y: np.ndarray,
                           config: dict) -> dict:
    pass  # complete here

# Test — valid run
X = np.random.randn(200, 10)
y = np.random.randint(0, 2, 200)
result = run_training_pipeline(X, y, {"min_auc": 0.4, "cv": 3})
if result:
    print("Result keys:", list(result.keys()))

# Test — invalid data
X_nan = X.copy(); X_nan[0, 0] = np.nan
try:
    run_training_pipeline(X_nan, y, {"min_auc": 0.4, "cv": 3})
except DataValidationError as e:
    print(f"Caught DataValidationError: {e}")

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def run_training_pipeline(X, y, config):
#     log_path = Path("/tmp/pipeline.log")
#     logger   = setup_logger("pipeline", log_path)
#     try:
#         # 1. Validate
#         logger.info(f"Validating data: shape={X.shape}")
#         if np.isnan(X).any():
#             raise DataValidationError(f"X contains {np.isnan(X).sum()} NaN values")
#
#         # 2. Preprocess
#         logger.info("Building pipeline")
#         pipe = Pipeline([
#             ("scale", StandardScaler()),
#             ("clf",   LogisticRegression(max_iter=1000, random_state=42)),
#         ])
#
#         # 3. Train + evaluate
#         logger.info(f"Running {config['cv']}-fold CV")
#         scores = cross_val_score(pipe, X, y, cv=config["cv"], scoring="roc_auc")
#         auc    = scores.mean()
#         logger.info(f"ROC-AUC: {auc:.4f} ± {scores.std():.4f}")
#
#         if auc < config["min_auc"]:
#             raise ModelTrainingError(f"AUC {auc:.4f} < min_auc {config['min_auc']}")
#
#         # 4. Final fit
#         pipe.fit(X, y)
#         logger.info("Training complete")
#         return {"model": pipe, "metrics": {"roc_auc": auc}, "log_path": log_path}
#
#     except DataValidationError as e:
#         logger.error(f"Data validation failed: {e}")
#         raise
#     except ModelTrainingError as e:
#         logger.error(f"Model training failed: {e}")
#         raise

---
## 📋 Module 3 Recap

| Topic | Test |
|---|---|
| BaseEstimator + TransformerMixin | Write a custom transformer from scratch with fit/transform |
| check_is_fitted | Know when and why to call it |
| get_params / set_params | Understand why __init__ param names matter |
| ABC | Define an interface with abstract methods + concrete shared methods |
| ColumnTransformer | Handle mixed numeric + categorical types in one step |
| Pipeline.set_params | Tune nested params using __ notation |
| clone | Create an unfitted copy of a pipeline |
| pathlib | Never use os.path — always Path, /, .stem, .suffix, .rglob |
| joblib | Serialize and reload a model with metadata as a bundle |
| Pydantic config | Load and validate config from JSON or env vars |
| ModelRegistry | Simple file-based registry with index.json |
| logging | setup_logger with file + console handlers |
| log_call decorator | Wrap any function with DEBUG/INFO/ERROR logging |
| Structured exceptions | DataValidationError, ModelTrainingError — log before re-raise |
